<a href="https://colab.research.google.com/github/imguru-mooc/AI_CAMPUS/blob/main/NN-day1/NN_Day1_lecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Network Day 1 — Python · NumPy와 AI 수학

**온디바이스 AI · Neural Network 1/5 · 10/20(화)** · Google Colab · Python + NumPy

오늘의 목표

1. Colab 노트북을 다루고, C 개발자 관점에서 Python 문법을 정리한다.
2. NumPy `ndarray` 의 **shape · axis · 브로드캐스팅**을 자유롭게 읽는다.
3. 벡터 · 행렬 연산과 **미분 · 기울기**를 NumPy 코드로 구현한다.
4. 8교시 미니 프로젝트에서 **경사하강법**으로 함수의 최소점을 찾는다.

| 교시 | 주제 |
|---|---|
| 1 | 과정 안내 · Colab 사용법 |
| 2 | Python 속성 (C 개발자용) |
| 3 | NumPy ndarray |
| 4 | 벡터화 |
| 5 | 선형대수 |
| 6 | 미분과 기울기 |
| 7 | 확률 기초 |
| 8 | 미니 프로젝트 — 경사하강법 (이 노트북 맨 끝) |

**노트북 사용법**

1. `____` 가 있는 셀은 빈칸을 채운 뒤 실행합니다.
2. 바로 아래 **확인 셀**이 `✅` 를 출력하면 정답입니다. `AssertionError` 가 나면 다시 생각해 보세요.
3. 막히면 그 아래 **🔒 정답 보기** 셀을 실행하면 정답 코드와 해설이 출력됩니다. 먼저 스스로 풀어 본 뒤에 여세요.
4. 8교시 미니 프로젝트도 이 노트북 맨 끝에 있습니다. 채점 셀 결과가 보이는 상태로 제출합니다.

### 📚 오늘의 온라인 자료

| 자료 | 링크 |
|---|---|
| 🎬 애니메이션 (시뮬레이션 9개) | [https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html) |
| 📝 개념 워크시트 (8교시 × 10문항) | [https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html) |
| 🖥 슬라이드 (PDF) | [https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/NN_Day1.pdf](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/NN_Day1.pdf) |

각 교시 첫머리에도 그 교시의 애니메이션 · 워크시트 바로가기가 있습니다.

In [ ]:
# ▶ 환경 셀 — 런타임이 끊기면 이 셀부터 다시 실행하세요
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5, 4)
rng = np.random.default_rng(42)          # 모두 같은 난수를 쓰도록 seed 고정

try:                                      # Colab 에서만 Drive 연결
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Colab 이 아닌 환경 — Drive 마운트 생략")

print("NumPy", np.__version__)

## 1교시 — 과정 안내 · Colab 사용법

### AI · 머신러닝 · 딥러닝

- **AI** : 사람의 지적 작업을 기계가 하도록 만드는 모든 기술
- **머신러닝** : 규칙을 사람이 쓰지 않고 **데이터에서 학습**하는 방법
- **딥러닝** : 여러 층의 **신경망(Neural Network)** 으로 학습하는 머신러닝

### 학습이란 무엇인가 — 이번 과정의 한 문장

> **학습 = 손실(Loss)을 가장 작게 만드는 파라미터를 찾는 최적화**

$$\theta^{*} = \arg\min_{\theta} L(\theta)$$

5일 동안 이 한 문장을 직접 코드로 만듭니다.

| 일차 | 만드는 것 |
|---|---|
| Day 1 | 도구: NumPy · 미분 · 경사하강 |
| Day 2 | 뉴런 · 순전파 · 손실 |
| Day 3 | 역전파 → MNIST 학습 |
| Day 4 | 옵티마이저 · 초기화 · BatchNorm · Dropout → `mynn.py` |
| Day 5 | 합성곱 · 풀링 → CNN |

### Colab 기본기

| 동작 | 단축키 |
|---|---|
| 셀 실행 | `Ctrl + Enter` |
| 실행 후 다음 셀로 | `Shift + Enter` |
| 위 / 아래에 코드 셀 추가 | `Ctrl + M A` / `Ctrl + M B` |
| 셀 삭제 | `Ctrl + M D` |
| 마크다운 ↔ 코드 전환 | `Ctrl + M M` / `Ctrl + M Y` |

- **런타임** : 노트북 뒤에서 돌아가는 Python 프로세스. 오래 쉬면 끊기고 **변수가 모두 사라집니다** → 환경 셀부터 다시 실행.
- **Drive에 사본 저장** : 파일 → Drive에 사본 저장. 사본에서 작업해야 내 코드가 남습니다.
- `!명령` 은 셸 명령, `%명령` 은 IPython 매직 명령입니다.

> 🎬 **애니메이션** [D1-1 학습은 최적화](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-1)  
> 📝 **워크시트** [1교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t1)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

### 그림으로 보는 AI · 머신러닝 · 딥러닝

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f1_ai_ml_dl.png" alt="AI ⊃ 머신러닝 ⊃ 딥러닝 포함 관계" width="760"></p>

- **AI** 는 가장 넓은 말입니다. 사람이 규칙을 직접 짠 체스 프로그램도 AI 입니다.
- **머신러닝**은 그중 "규칙을 사람이 쓰지 않고 데이터에서 찾는" 방법입니다.
- **딥러닝**은 머신러닝 중에서 **층(layer)을 여러 겹 쌓은 신경망**을 쓰는 방법입니다. 층이 깊을수록 이미지 · 음성처럼 규칙을 말로 쓰기 어려운 문제를 풀 수 있습니다.

온디바이스 AI 과정에서 앞으로 다룰 YOLO · 센서 이상 감지 모델은 모두 가장 안쪽 원, **딥러닝**에 속합니다.

### 규칙을 쓰는가, 데이터에서 찾는가

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f1_rule_vs_learn.png" alt="전통 프로그래밍과 머신러닝 비교" width="760"></p>

섭씨를 화씨로 바꾸는 규칙 `F = 1.8C + 32` 는 사람이 알고 있으니 코드로 쓰면 됩니다(왼쪽). 하지만 "사진 속 사람이 헬멧을 썼는가" 같은 규칙은 말로 쓸 수 없습니다. 이럴 때 **입력과 정답 쌍을 잔뜩 주고 컴퓨터가 규칙(숫자 w, b)을 찾게** 하는 것이 머신러닝입니다(오른쪽).

신경망이 찾는 "규칙"은 결국 **숫자(파라미터)의 모음**입니다. 화씨 예제는 숫자 2개(w, b), 오늘 8교시 미니 프로젝트는 2개, Day 3 MNIST 신경망은 약 8만 개, YOLOv8n 은 약 300만 개입니다.

### 학습의 고리 — 5일 동안 만들 전체 그림

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f1_learning_loop.png" alt="데이터 → 모델 → 손실 → 기울기 → 갱신 반복" width="720"></p>

| 고리의 단계 | 하는 일 | 이 과정에서 만드는 날 |
|---|---|---|
| 모델 y = f(x; θ) | 입력에 파라미터를 곱하고 더해 예측 | Day 2 순전파 |
| 손실 L | 예측과 정답의 차이를 숫자 하나로 | Day 2 MSE · Cross-Entropy |
| 기울기 ∂L/∂θ | 파라미터를 어느 쪽으로 바꿔야 손실이 줄어드나 | **Day 1 미분**, Day 3 역전파 |
| 갱신 θ ← θ − η∇L | 기울기 반대 방향으로 조금 이동 | **Day 1 경사하강**, Day 4 옵티마이저 |

오늘은 이 고리를 돌리는 **도구**(NumPy · 행렬 · 미분)를 만들고, 8교시에 고리를 처음으로 직접 돌려 봅니다.

#### ⚠️ Colab 에서 자주 하는 실수

- **원본에서 작업** — GitHub 에서 연 노트북은 저장되지 않습니다. 반드시 *파일 → Drive에 사본 저장* 후 작업하세요.
- **셀 실행 순서 꼬임** — 셀 왼쪽 `[번호]` 가 실제 실행 순서입니다. 위 셀을 고치고 다시 실행하지 않으면 예전 값이 남아 있습니다. 헷갈리면 *런타임 → 다시 시작 후 모두 실행*.
- **런타임 끊김 후 NameError** — 변수가 사라진 것입니다. 맨 위 환경 셀부터 다시 실행하세요.

In [ ]:
!python --version
!nproc            # Colab CPU 코어 수

In [ ]:
# %timeit : 코드를 여러 번 돌려 평균 실행 시간을 잰다
%timeit sum(range(1000))

**연습 1-1** `%timeit` 으로 `sum(range(1_000_000))` 의 실행 시간을 재 보세요.

In [ ]:
%timeit ____

In [ ]:
#@title 🔒 정답 보기 1-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAxLTEg4pSA4pSACiV0aW1laXQgc3VtKHJhbmdlKDFfMDAwXzAwMCkp"
).decode("utf-8"))

#### 📌 1교시 정리

- 학습 = **손실을 최소로 만드는 파라미터 찾기**. 신경망의 규칙은 숫자(파라미터)의 모음이다.
- 학습의 고리: 모델 → 손실 → 기울기 → 갱신 → 반복.
- Colab: 사본에서 작업, 실행 순서는 `[번호]`, 끊기면 환경 셀부터.

## 2교시 — Python 속성 (C 개발자용)

| C | Python | 핵심 차이 |
|---|---|---|
| `int a = 3;` | `a = 3` | 변수 = 객체를 가리키는 **이름**, 타입은 객체에 붙어 있다 (동적 타입) |
| `int arr[5];` | `arr = [0]*5` | 리스트는 길이가 늘고 줄며 타입이 섞여도 된다 |
| `struct` | `class` | 메서드를 함께 묶는다, `self` = C 의 `this` 포인터 |
| `{ }` 블록 | 들여쓰기 | 들여쓰기 자체가 문법 |
| `for (i=0;i<n;i++)` | `for i in range(n):` | 반복 가능한 객체를 순회 |
| 포인터 | 참조 | `b = a` 는 **복사가 아니라** 같은 객체를 가리킨다 |

> 🎬 **애니메이션** [D1-2 이름과 객체 모델](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-2)  
> 📝 **워크시트** [2교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t2)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

### C 의 변수 vs Python 의 이름

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f2_names_objects.png" alt="이름이 객체를 가리키는 Python 메모리 모델" width="680"></p>

C 에서 `int a = 3;` 은 **3 이 들어 있는 메모리 칸**입니다. Python 에서 `a = 3` 은 heap 어딘가에 있는 **int 객체 3 에 `a` 라는 이름표를 붙이는 것**입니다. 그래서

- `b = a` 는 값을 복사하지 않고 **같은 객체에 이름표를 하나 더** 붙입니다.
- 리스트처럼 **바뀔 수 있는(mutable)** 객체는 `b.append(4)` 가 `a` 에도 보입니다.
- int · float · str · tuple 은 **바뀔 수 없는(immutable)** 객체라서 `b = b + 1` 은 기존 객체를 고치지 않고 **새 객체**를 만들어 `b` 를 옮겨 붙입니다.

| 구분 | 예 | `b = a` 후 b 를 바꾸면 |
|---|---|---|
| 불변 (immutable) | int, float, str, tuple | a 는 그대로 |
| 가변 (mutable) | list, dict, **NumPy ndarray** | a 도 바뀜 (같은 객체) |

In [ ]:
# is : 같은 객체인가?   == : 값이 같은가?
a = [1, 2, 3]
b = a
c = a.copy()
print("a is b :", a is b, "| id 같음:", id(a) == id(b))
print("a is c :", a is c, "| 값은 같음:", a == c)

x = 3
y = x
y = y + 1          # int 는 불변 → 새 객체 4 를 만들어 y 를 옮김
print("x =", x, " y =", y)

### C 배열 vs Python 리스트 — 왜 NumPy 가 필요한가

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f2_c_vs_py.png" alt="C 배열은 값이 연속, Python 리스트는 참조의 배열" width="760"></p>

Python 리스트의 칸에는 값이 아니라 **객체를 가리키는 참조**가 들어 있습니다. 덕분에 길이가 자유롭고 타입이 섞여도 되지만, 숫자 하나를 더할 때마다 "참조 따라가기 → 타입 확인 → 덧셈 → 새 객체 만들기"를 반복하므로 느립니다. 3교시의 **NumPy ndarray 는 C 배열처럼 값을 연속으로** 저장해 이 문제를 해결합니다.

In [ ]:
# 동적 타입 — 같은 이름이 다른 타입의 객체를 가리킬 수 있다
x = 3;      print(type(x))
x = 3.0;    print(type(x))
x = "three"; print(type(x))

# 정수는 크기 제한이 없다 (C 의 overflow 가 없음)
print(2 ** 100)

In [ ]:
# 리스트 · 딕셔너리 · 함수 · 클래스
nums = [3, 1, 4, 1, 5]
nums.append(9)
print(nums, len(nums), nums[-1], nums[1:3])

price = {"apple": 100, "orange": 150}
price["banana"] = 80
for k, v in price.items():
    print(k, v)

def add(a, b=10):          # 기본값 인자
    return a + b
print(add(1), add(1, 2))

class Counter:
    def __init__(self):    # 생성자
        self.n = 0
    def inc(self):
        self.n += 1
        return self.n
c = Counter(); c.inc(); print(c.inc())

### C → Python 바꿔 쓰기 (5문제)

**문제 2-1** C 코드 `for(i=1;i<=100;i++) total += i;` 를 한 줄로

In [ ]:
total = ____

In [ ]:
assert total == 5050; print("✅")

In [ ]:
#@title 🔒 정답 보기 2-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTEg4pSA4pSACnRvdGFsID0gc3VtKHJhbmdlKDEsIDEwMSkp"
).decode("utf-8"))

**문제 2-2** C 의 임시 변수 `t` 없이 두 값 교환 (`t=a; a=b; b=t;`)

In [ ]:
a, b = 1, 2
a, b = ____

In [ ]:
assert (a, b) == (2, 1); print("✅")

In [ ]:
#@title 🔒 정답 보기 2-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTIg4pSA4pSACmEsIGIgPSAxLCAyCmEsIGIgPSBiLCBh"
).decode("utf-8"))

**문제 2-3** C 구조체 `struct Point {double x, y;};` 와 원점 거리 함수를 클래스로

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def dist(self):
        return ____

In [ ]:
assert Point(3, 4).dist() == 5.0; print("✅")

In [ ]:
#@title 🔒 정답 보기 2-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTMg4pSA4pSACmNsYXNzIFBvaW50OgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHgsIHkp"
    "OgogICAgICAgIHNlbGYueCA9IHgKICAgICAgICBzZWxmLnkgPSB5CiAgICBkZWYgZGlzdChzZWxmKToKICAgICAg"
    "ICByZXR1cm4gKHNlbGYueCAqKiAyICsgc2VsZi55ICoqIDIpICoqIDAuNQ=="
).decode("utf-8"))

**문제 2-4** 참조 — C 에서 포인터 두 개가 같은 배열을 가리키는 상황과 같다. `b` 에 추가하면 `a` 는?

In [ ]:
a = [1, 2, 3]
b = a              # 복사가 아니다!
b.append(4)
c = a.____      # 진짜 복사본
c.append(5)
answer_a = ____

In [ ]:
assert a == answer_a and c == [1, 2, 3, 4, 5]; print("✅")

In [ ]:
#@title 🔒 정답 보기 2-4 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTQg4pSA4pSACmEgPSBbMSwgMiwgM10KYiA9IGEgICAgICAgICAgICAgICMg67O17IKs"
    "6rCAIOyVhOuLiOuLpCEKYi5hcHBlbmQoNCkKYyA9IGEuY29weSgpICAgICAgIyDsp4Tsp5wg67O17IKs67O4CmMu"
    "YXBwZW5kKDUpCmFuc3dlcl9hID0gWzEsIDIsIDMsIDRdCgrwn5KhIO2VtOyEpApiID0gYSDripQg6rCZ7J2AIOum"
    "rOyKpO2KuCDqsJ3ssrTsl5Ag7J2066aE7J2EIO2VmOuCmCDrjZQg67aZ7J24IOqyg+ydtOudvCBiLmFwcGVuZCDq"
    "sIAgYSDsl5Drj4Qg67O07J2464ukLiBOdW1QeSDrsLDsl7Trj4Qg6rCZ7J2AIOq3nOy5meydhCDrlLDrpbTrr4Dr"
    "oZwgRGF5IDMg7Jet7KCE7YyM7JeQ7IScIFcuY29weSgpIOulvCDrubzrqLnsnLzrqbQg7JuQ67O4IOqwgOykkey5"
    "mOqwgCDrsJTrgIzripQg67KE6re46rCAIOyDneq4tOuLpC4="
).decode("utf-8"))

**문제 2-5** 단어 빈도 세기 — C 라면 해시 테이블을 직접 만들어야 한다. 딕셔너리 `get(키, 기본값)` 을 써라.

In [ ]:
words = "cat dog cat bird dog cat".split()
counts = {}
for w in words:
    counts[w] = counts.____ + 1

In [ ]:
assert counts == {"cat": 3, "dog": 2, "bird": 1}; print("✅")

In [ ]:
#@title 🔒 정답 보기 2-5 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTUg4pSA4pSACndvcmRzID0gImNhdCBkb2cgY2F0IGJpcmQgZG9nIGNhdCIuc3BsaXQo"
    "KQpjb3VudHMgPSB7fQpmb3IgdyBpbiB3b3JkczoKICAgIGNvdW50c1t3XSA9IGNvdW50cy5nZXQodywgMCkgKyAx"
).decode("utf-8"))

**보너스** 리스트 컴프리헨션 — 0~9 중 짝수의 제곱 리스트를 한 줄로

In [ ]:
squares = [____]

In [ ]:
assert squares == [0, 4, 16, 36, 64]; print("✅")

In [ ]:
#@title 🔒 정답 보기 2-6 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTYg4pSA4pSACnNxdWFyZXMgPSBbeCAqKiAyIGZvciB4IGluIHJhbmdlKDEwKSBpZiB4"
    "ICUgMiA9PSAwXQ=="
).decode("utf-8"))

#### ⚠️ Python 에서 C 개발자가 자주 하는 실수

- `b = a` 를 **복사**라고 생각함 → 리스트 · ndarray 는 `.copy()`.
- `/` 는 항상 실수 나눗셈 (`7 / 2 = 3.5`), 정수 몫은 `//` (`7 // 2 = 3`).
- 음수 인덱스 `a[-1]` 은 **마지막 원소** — 오류가 아닙니다.
- 들여쓰기에 탭과 스페이스를 섞으면 `IndentationError`.

#### 📌 2교시 정리

- 변수 = 객체에 붙인 **이름표**, 타입은 객체에 있다(동적 타입).
- 가변 객체(list, ndarray)는 `b = a` 로 공유된다 → 독립 사본은 `.copy()`.
- Python 리스트는 참조의 배열이라 숫자 계산에 느리다 → NumPy.

## 3교시 — NumPy ndarray

`ndarray` = **같은 타입의 숫자가 연속 메모리에 놓인 다차원 배열** (C 의 `double a[3][4]` 와 같은 구조). 신경망의 입력 · 가중치 · 출력은 전부 ndarray 입니다.

| 속성 | 의미 | 예: `np.zeros((3, 4))` |
|---|---|---|
| `ndim` | 차원 수 | 2 |
| `shape` | 각 차원 크기 | `(3, 4)` |
| `size` | 원소 수 | 12 |
| `dtype` | 원소 타입 | `float64` |

> 🎬 **애니메이션** [D1-3 ndarray 메모리 · 브로드캐스팅](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-3) · [D1-4 axis 축소 · keepdims](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-4)  
> 📝 **워크시트** [3교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t3)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

### ndarray 의 정체 — 한 줄 메모리 + 읽는 규칙

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f3_memory_strides.png" alt="연속 메모리와 strides" width="760"></p>

`np.arange(12).reshape(3, 4)` 는 숫자 12개를 **메모리에 한 줄로** 놓고, `shape=(3, 4)` 와 `strides=(32, 8)` 라는 **읽는 규칙**으로 2차원처럼 보여 줍니다.

- **strides** = 각 축 방향으로 한 칸 움직일 때 건너뛰는 바이트 수. int64 는 원소 하나가 8바이트이므로 열 방향 8, 행 방향 4×8 = 32.
- `A[i, j]` 의 위치 = `i × 32 + j × 8` 바이트. C 의 2차원 배열 주소 계산과 똑같습니다.
- `reshape` 는 대부분 **메모리를 복사하지 않고 규칙만 바꾼 view** 를 돌려줍니다. 그래서 view 를 고치면 원본도 바뀝니다.

In [ ]:
A = np.arange(12).reshape(3, 4)
print("shape:", A.shape, " strides:", A.strides, " itemsize:", A.itemsize, " dtype:", A.dtype)

B = A.reshape(4, 3)          # view — 메모리 공유
B[0, 0] = 100
print("B[0,0] 을 바꾼 뒤 A[0,0] =", A[0, 0])       # 100 — 원본도 바뀜
print("메모리 공유?", np.shares_memory(A, B))

C = A.reshape(4, 3).copy()   # 독립 사본
C[0, 0] = -1
print("C 를 바꾼 뒤 A[0,0] =", A[0, 0])            # 그대로 100
A[0, 0] = 0                  # 원래 값으로 되돌리기

In [ ]:
a = np.array([1, 2, 3])
B = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=np.float32)
print(a.ndim, a.shape, a.dtype)
print(B.ndim, B.shape, B.dtype)

print(np.zeros((2, 3)))
print(np.ones(3))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))
print(rng.normal(size=(2, 2)))

### 인덱싱 · 슬라이싱 · reshape

In [ ]:
A = np.arange(12).reshape(3, 4)
print(A)
print("A[1, 2]   =", A[1, 2])      # 1행 2열
print("A[0]      =", A[0])         # 0행 전체
print("A[:, 1]   =", A[:, 1])      # 1열 전체
print("A[1:, 2:] =\n", A[1:, 2:])  # 부분 행렬
print("A[A > 6]  =", A[A > 6])     # 불리언 마스크 — Day 3 ReLU 역전파에서 사용

### axis — 어느 방향으로 줄이는가

`A.sum(axis=0)` 은 **axis 0(행 방향)을 없앤다** → 열마다 합. `shape (3, 4) → (4,)`

`A.sum(axis=1)` 은 **axis 1(열 방향)을 없앤다** → 행마다 합. `shape (3, 4) → (3,)`

### 브로드캐스팅 규칙

두 배열의 shape 를 **오른쪽 끝부터** 비교해, 각 차원이 ① 같거나 ② 한쪽이 1 이면 연산 가능. 1 인 쪽이 늘어난다.

```
(3, 4) + (4,)    →  (3, 4) + (1, 4)  →  (3, 4)
(3, 1) + (1, 4)  →  (3, 4)
(3, 4) + (3,)    →  ❌  (4 와 3 이 다르다)
```

#### 그림으로 보는 axis

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f3_ndarray_axes.png" alt="axis 0 과 axis 1 방향의 합" width="700"></p>

**axis = 없어지는 축**이라고 기억하세요. `sum(axis=0)` 은 행 방향(위→아래)으로 더해 axis 0 이 사라지고 열마다 하나씩 남습니다. 신경망에서는 **배치 축(axis 0)** 으로 평균을 내는 일이 아주 많습니다(Day 3 손실 평균, Day 4 BatchNorm).

#### 그림으로 보는 브로드캐스팅

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f3_broadcast.png" alt="(3,4) + (4,) 브로드캐스팅" width="760"></p>

`b` 가 실제로 3번 복사되는 것은 아니고, NumPy 가 **복사한 것처럼** 계산합니다(메모리 절약). 신경망 한 층의 `X @ W + b` 에서 편향 `b` 가 모든 샘플(행)에 더해지는 것이 바로 이 그림입니다.

In [ ]:
A = np.arange(12).reshape(3, 4)
b = np.array([10, 20, 30, 40])
print(A + b)                     # 모든 행에 b 를 더한다 — 신경망의 "+ 편향"

### shape 맞히기 10문제

`A = np.zeros((3, 4))` 일 때 각 결과의 shape 를 튜플로 쓰세요.

In [ ]:
A = np.zeros((3, 4))
answers = {
    "A.T":                         ____,
    "A.reshape(2, 6)":             ____,
    "A.reshape(-1)":               ____,
    "A[0]":                        ____,
    "A[:, 1:3]":                   ____,
    "A.sum(axis=0)":               ____,
    "A.sum(axis=1)":               ____,
    "A.sum(axis=1, keepdims=True)": ____,
    "A + np.ones(4)":              ____,
    "np.ones((3, 1)) + np.ones((1, 4))": ____,
}

In [ ]:
for expr, ans in answers.items():
    real = eval(expr).shape
    print(f"{'✅' if real == ans else '❌'} {expr:35s} 정답 {real}  내 답 {ans}")
assert all(eval(e).shape == s for e, s in answers.items())

In [ ]:
#@title 🔒 정답 보기 3-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAzLTEg4pSA4pSACkEgPSBucC56ZXJvcygoMywgNCkpCmFuc3dlcnMgPSB7CiAgICAiQS5U"
    "IjogICAgICAgICAgICAgICAgICAgICAgICAgKDQsIDMpLAogICAgIkEucmVzaGFwZSgyLCA2KSI6ICAgICAgICAg"
    "ICAgICgyLCA2KSwKICAgICJBLnJlc2hhcGUoLTEpIjogICAgICAgICAgICAgICAoMTIsKSwKICAgICJBWzBdIjog"
    "ICAgICAgICAgICAgICAgICAgICAgICAoNCwpLAogICAgIkFbOiwgMTozXSI6ICAgICAgICAgICAgICAgICAgICgz"
    "LCAyKSwKICAgICJBLnN1bShheGlzPTApIjogICAgICAgICAgICAgICAoNCwpLAogICAgIkEuc3VtKGF4aXM9MSki"
    "OiAgICAgICAgICAgICAgICgzLCksCiAgICAiQS5zdW0oYXhpcz0xLCBrZWVwZGltcz1UcnVlKSI6ICgzLCAxKSwK"
    "ICAgICJBICsgbnAub25lcyg0KSI6ICAgICAgICAgICAgICAoMywgNCksCiAgICAibnAub25lcygoMywgMSkpICsg"
    "bnAub25lcygoMSwgNCkpIjogKDMsIDQpLAp9Cgrwn5KhIO2VtOyEpAprZWVwZGltcz1UcnVlIOuKlCDspITsnbgg"
    "7LaV7J2EIO2BrOq4sCAx66GcIOuCqOq4tOuLpC4gKDMsIDEpIOydgCDri6Tsi5wgKDMsIDQpIOyZgCDruIzroZzr"
    "k5zsupDsiqTtjIXrkJjrr4DroZwgIu2WieuniOuLpCDtj4nqt6DsnYQg67m86riwIiDqsJnsnYAg7Jew7IKw7JeQ"
    "IO2VhOyImOuLpCAoRGF5IDIgU29mdG1heCDsiJjsuZgg7JWI7KCV7ZmUKS4="
).decode("utf-8"))

#### ⚠️ ndarray 에서 자주 하는 실수

- `np.array([1, 2, 3])` 은 **int64** — 여기에 0.5 를 대입하면 0 으로 잘립니다. 실수 계산은 `dtype=float` 또는 `[1., 2., 3.]`.
- 슬라이스 · reshape 는 **view** — 고치면 원본도 바뀝니다. 필요하면 `.copy()`.
- `(3,)` 와 `(3, 1)` 은 다른 shape — 1차원 벡터는 행도 열도 아닙니다.

#### 📌 3교시 정리

- ndarray = 같은 dtype 의 **연속 메모리** + shape · strides 규칙.
- `axis` = 없어지는 축, `keepdims=True` 로 크기 1 을 남겨 다시 브로드캐스팅.
- 브로드캐스팅: **오른쪽 축부터** 같거나 1 이면 결합.

## 4교시 — 벡터화

**벡터화** = Python `for` 루프 대신 NumPy 연산 한 번으로 배열 전체를 처리하는 것. NumPy 내부는 C · BLAS 로 짜여 있어 수십~수천 배 빠릅니다. 신경망은 수백만 번의 곱셈-덧셈이므로 벡터화가 필수입니다.

> 🎬 **애니메이션** [D1-5 행렬곱 추적기](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-5)  
> 📝 **워크시트** [4교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t4)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

In [ ]:
x = np.arange(5)
print(x * 2, x ** 2, np.sqrt(x), np.exp(x))   # 원소별 연산 — 루프 없음

### 행렬곱 한 칸은 "행 · 열" 내적

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f4_matmul.png" alt="C[1][0] = A 의 1행과 B 의 0열의 내적" width="760"></p>

`C = A @ B` 의 칸 `C[i][j]` 는 **A 의 i 행과 B 의 j 열을 원소끼리 곱해 더한 값**(내적)입니다. A 의 열 수와 B 의 행 수가 같아야 짝이 맞고, 그 축은 계산 후 사라집니다. 아래 3중 for 루프는 이 정의를 그대로 옮긴 것입니다.

### 행렬곱 for 루프 vs `@`

**문제 4-1** 3중 for 루프 행렬곱을 완성하세요. $C_{ij} = \sum_k A_{ik} B_{kj}$

In [ ]:
def matmul_loop(A, B):
    n, m = len(A), len(A[0])
    p = len(B[0])
    C = [[0.0] * p for _ in range(n)]
    for i in range(n):
        for j in range(p):
            for k in range(m):
                C[i][j] += ____
    return C

In [ ]:
A = rng.random((3, 4)); B = rng.random((4, 2))
assert np.allclose(matmul_loop(A.tolist(), B.tolist()), A @ B); print("✅")

In [ ]:
#@title 🔒 정답 보기 4-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTEg4pSA4pSACmRlZiBtYXRtdWxfbG9vcChBLCBCKToKICAgIG4sIG0gPSBsZW4oQSks"
    "IGxlbihBWzBdKQogICAgcCA9IGxlbihCWzBdKQogICAgQyA9IFtbMC4wXSAqIHAgZm9yIF8gaW4gcmFuZ2Uobild"
    "CiAgICBmb3IgaSBpbiByYW5nZShuKToKICAgICAgICBmb3IgaiBpbiByYW5nZShwKToKICAgICAgICAgICAgZm9y"
    "IGsgaW4gcmFuZ2UobSk6CiAgICAgICAgICAgICAgICBDW2ldW2pdICs9IEFbaV1ba10gKiBCW2tdW2pdCiAgICBy"
    "ZXR1cm4gQw=="
).decode("utf-8"))

In [ ]:
N = 200
A = rng.random((N, N)); B = rng.random((N, N))
Al, Bl = A.tolist(), B.tolist()

t0 = time.perf_counter(); matmul_loop(Al, Bl); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); A @ B;              t_np   = time.perf_counter() - t0
print(f"for 루프 : {t_loop*1000:9.1f} ms")
print(f"A @ B    : {t_np*1000:9.3f} ms")
print(f"속도 차  : {t_loop / t_np:,.0f} 배")

In [ ]:
%timeit A @ B

### 왜 이렇게 차이가 날까?

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f4_vectorize.png" alt="for 루프와 @ 의 실행 시간 비교" width="620"></p>

| 순수 Python for 루프 | NumPy `@` |
|---|---|
| 곱셈 한 번마다 인터프리터가 명령 해석 | C · Fortran 으로 컴파일된 BLAS 함수 한 번 호출 |
| 원소마다 타입 확인 · 객체 생성 | 같은 dtype 이라 확인 불필요 |
| 참조를 따라 메모리 여기저기 접근 | 연속 메모리를 캐시 · SIMD 로 한꺼번에 처리 |

200×200 행렬곱은 곱셈이 800만 번입니다. Day 3 MNIST 신경망은 한 번 학습할 때 이런 곱셈을 **수십억 번** 하므로, 벡터화하지 않으면 실습 시간 안에 끝나지 않습니다.

### axis 를 쓰는 집계 함수

학생 5명 × 과목 3개 점수표로 연습합니다. `scores[i, j]` = i 번 학생의 j 과목 점수.

In [ ]:
scores = np.array([[90, 70, 80],
                   [60, 95, 85],
                   [75, 80, 70],
                   [88, 92, 95],
                   [50, 65, 60]])

subject_mean = scores.mean(axis=____)          # 과목별 평균 → shape (3,)
student_total = scores.sum(axis=____)          # 학생별 총점 → shape (5,)
best_student = ____     # 총점 1등 학생 번호
centered = scores - scores.mean(axis=1, ____)  # 학생별 평균을 뺀 점수

In [ ]:
assert subject_mean.shape == (3,) and np.allclose(subject_mean, [72.6, 80.4, 78.0])
assert student_total.tolist() == [240, 240, 225, 275, 175]
assert best_student == 3
assert np.allclose(centered.mean(axis=1), 0)
print("✅")

In [ ]:
#@title 🔒 정답 보기 4-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTIg4pSA4pSACnNjb3JlcyA9IG5wLmFycmF5KFtbOTAsIDcwLCA4MF0sCiAgICAgICAg"
    "ICAgICAgICAgICBbNjAsIDk1LCA4NV0sCiAgICAgICAgICAgICAgICAgICBbNzUsIDgwLCA3MF0sCiAgICAgICAg"
    "ICAgICAgICAgICBbODgsIDkyLCA5NV0sCiAgICAgICAgICAgICAgICAgICBbNTAsIDY1LCA2MF1dKQoKc3ViamVj"
    "dF9tZWFuID0gc2NvcmVzLm1lYW4oYXhpcz0wKSAgICAgICAgICAjIOqzvOuqqeuzhCDtj4nqt6Ag4oaSIHNoYXBl"
    "ICgzLCkKc3R1ZGVudF90b3RhbCA9IHNjb3Jlcy5zdW0oYXhpcz0xKSAgICAgICAgICAjIO2VmeyDneuzhCDstJ3s"
    "oJAg4oaSIHNoYXBlICg1LCkKYmVzdF9zdHVkZW50ID0gbnAuYXJnbWF4KHN0dWRlbnRfdG90YWwpICAgICAjIOy0"
    "neygkCAx65OxIO2VmeyDnSDrsojtmLgKY2VudGVyZWQgPSBzY29yZXMgLSBzY29yZXMubWVhbihheGlzPTEsIGtl"
    "ZXBkaW1zPVRydWUpICAjIO2VmeyDneuzhCDtj4nqt6DsnYQg67qAIOygkOyImAoK8J+SoSDtlbTshKQKYXJnbWF4"
    "IOuKlCDsi6Dqsr3rp50g7Lac66ClKO2BtOuemOyKpOuzhCDsoJDsiJgp7JeQ7IScIOyYiOy4oSDtgbTrnpjsiqTr"
    "pbwg6rOg66W8IOuVjCDqt7jrjIDroZwg7JO064ukOiB5X3ByZWQgPSBucC5hcmdtYXgoc2NvcmVzLCBheGlzPTEp"
    "Lg=="
).decode("utf-8"))

#### ⚠️ 자주 하는 실수

- `A * B` 는 **원소별 곱**, 행렬곱은 `A @ B` (또는 `np.dot`). shape 가 같으면 `*` 도 오류 없이 돌아가서 버그를 찾기 어렵습니다.
- 행렬곱 오류 메시지 `matmul: Input operand 1 has a mismatch in its core dimension` → 안쪽 차원이 다르다는 뜻. shape 부터 출력하세요.

#### 📌 4교시 정리

- 벡터화 = 반복문을 NumPy 내부(C · BLAS)로 넘기기 → 수백 배 빠름.
- `C[i][j]` = A 의 i 행 · B 의 j 열 내적, `(N, D) @ (D, H) → (N, H)`.
- `mean · sum · argmax` 는 axis 로 "무엇을 기준으로" 집계할지 정한다.

## 5교시 — 선형대수

### 벡터 내적 · 노름

$$\mathbf{a}\cdot\mathbf{b} = \sum_i a_i b_i = \lVert\mathbf{a}\rVert\,\lVert\mathbf{b}\rVert\cos\theta$$

- 내적은 **두 벡터가 얼마나 같은 방향인가**를 잰다. 뉴런의 가중합 $\mathbf{w}\cdot\mathbf{x}$ 도 내적이다.
- 노름 $\lVert\mathbf{a}\rVert = \sqrt{\mathbf{a}\cdot\mathbf{a}}$ = 벡터의 길이.

> 🎬 **애니메이션** [D1-6 2×2 선형 변환](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-6) · [D1-5 행렬곱 추적기](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-5)  
> 📝 **워크시트** [5교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t5)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

#### 그림으로 보는 내적

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f5_dot.png" alt="내적과 투영" width="560"></p>

내적은 "**b 를 a 방향으로 내린 그림자 길이 × a 의 길이**"입니다. 두 벡터가 같은 방향이면 크고, 직각이면 0, 반대면 음수입니다. 뉴런 하나는 입력 x 와 가중치 w 의 내적 `w · x` 를 계산하므로, **입력이 가중치 패턴과 얼마나 닮았는지**를 재는 장치라고 볼 수 있습니다.

In [ ]:
a = np.array([1.0, 2.0, 2.0])
b = np.array([2.0, 0.0, 1.0])
print("a·b   =", a @ b, np.dot(a, b))
print("|a|   =", np.linalg.norm(a))

**문제 5-1** 코사인 유사도 함수를 완성하세요.

In [ ]:
def cos_sim(a, b):
    return ____

In [ ]:
assert np.isclose(cos_sim(np.array([1., 0.]), np.array([0., 1.])), 0)
assert np.isclose(cos_sim(np.array([1., 1.]), np.array([2., 2.])), 1)
assert np.isclose(cos_sim(np.array([1., 0.]), np.array([-1., 0.])), -1)
print("✅")

In [ ]:
#@title 🔒 정답 보기 5-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA1LTEg4pSA4pSACmRlZiBjb3Nfc2ltKGEsIGIpOgogICAgcmV0dXJuIChhIEAgYikgLyAo"
    "bnAubGluYWxnLm5vcm0oYSkgKiBucC5saW5hbGcubm9ybShiKSk="
).decode("utf-8"))

### 행렬곱 = 선형 변환

행렬을 곱하면 공간이 **회전 · 확대 · 찌그러짐**됩니다. 신경망의 한 층 $XW$ 는 데이터를 새 공간으로 옮기는 변환입니다.

$$R(\theta) = \begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix},\qquad S = \begin{bmatrix}s_x & 0\\ 0 & s_y\end{bmatrix}$$

**문제 5-2** 회전 행렬을 완성하세요. 점들은 행 하나가 점 하나인 `(N, 2)` 배열이므로 `P @ R.T` 로 변환합니다.

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f5_transform.png" alt="행렬로 격자를 변환하기" width="760"></p>

행렬의 **열**은 기준 벡터 e₁ = (1, 0), e₂ = (0, 1) 이 옮겨 가는 위치입니다. 기준 벡터가 옮겨 가면 격자 전체가 따라 움직입니다. 넓이가 몇 배가 되는지는 **행렬식(det)** 이 알려 주고, det = 0 이면 평면이 직선으로 납작해져 정보가 사라집니다.

In [ ]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, ____],
                     [____, c]])

S = np.array([[2.0, 0.0],
              [0.0, 0.5]])

In [ ]:
assert np.allclose(np.array([1., 0.]) @ rot(np.pi / 2).T, [0., 1.]); print("✅")

In [ ]:
#@title 🔒 정답 보기 5-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA1LTIg4pSA4pSACmRlZiByb3QodGhldGEpOgogICAgYywgcyA9IG5wLmNvcyh0aGV0YSks"
    "IG5wLnNpbih0aGV0YSkKICAgIHJldHVybiBucC5hcnJheShbW2MsIC1zXSwKICAgICAgICAgICAgICAgICAgICAg"
    "W3MsIGNdXSkKClMgPSBucC5hcnJheShbWzIuMCwgMC4wXSwKICAgICAgICAgICAgICBbMC4wLCAwLjVdXSk="
).decode("utf-8"))

In [ ]:
# 정사각형 + 화살표 모양 점들
P = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0], [0.5, 1.5], [1, 1]], dtype=float)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for axi, (M, title) in zip(ax, [(rot(np.pi / 6), "rotate 30°"), (S, "scale (2, 0.5)"), (S @ rot(np.pi / 6), "rotate → scale")]):
    Q = P @ M.T
    axi.plot(P[:, 0], P[:, 1], "o-", label="before", alpha=.4)
    axi.plot(Q[:, 0], Q[:, 1], "o-", label="after")
    axi.set_title(title); axi.axis("equal"); axi.grid(alpha=.3); axi.legend()
plt.show()

### 행렬곱의 shape 규칙 — 신경망 한 층 미리 보기

$$\underbrace{X}_{(N,\,D)}\;\underbrace{W}_{(D,\,H)} + \underbrace{b}_{(H,)} = \underbrace{Y}_{(N,\,H)}$$

안쪽 차원 `D` 가 같아야 곱할 수 있고, 결과는 바깥쪽 `(N, H)` 입니다. **문제 5-3** 입력 4개 샘플 × 특징 3개를 출력 2개로 보내는 층을 만드세요.

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f5_layer.png" alt="X @ W + b = Y shape 블록 그림" width="760"></p>

In [ ]:
X = rng.normal(size=(4, 3))
W = rng.normal(size=(____))
b = np.zeros(____)
Y = ____

In [ ]:
assert W.shape == (3, 2) and b.shape == (2,) and Y.shape == (4, 2); print("✅", Y.shape)

In [ ]:
#@title 🔒 정답 보기 5-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA1LTMg4pSA4pSAClggPSBybmcubm9ybWFsKHNpemU9KDQsIDMpKQpXID0gcm5nLm5vcm1h"
    "bChzaXplPSgzLCAyKSkKYiA9IG5wLnplcm9zKDIpClkgPSBYIEAgVyArIGI="
).decode("utf-8"))

#### ⚠️ 자주 하는 실수

- 점을 행으로 쌓은 `P (N, 2)` 에는 `P @ M.T` — `M @ P` 는 shape 가 맞지 않습니다.
- 1차원 벡터 `(3,)` 끼리 `@` 하면 **스칼라(내적)** 가 나옵니다. 행렬이 필요하면 `reshape(1, 3)` 처럼 2차원으로.

#### 📌 5교시 정리

- 내적 = 같은 방향일수록 큰 값, 뉴런의 가중합.
- 행렬곱 = 공간의 선형 변환(회전 · 확대 · 찌그러짐), 행렬의 열 = 기준 벡터가 가는 곳.
- 신경망 한 층: `X (N, D) @ W (D, H) + b (H,) = Y (N, H)`.

## 6교시 — 미분과 기울기

### 미분 = 순간 변화율

$$f'(x) = \lim_{h\to 0}\frac{f(x+h)-f(x)}{h}$$

컴퓨터에서는 아주 작은 $h$ 로 근사합니다. 오차가 더 작은 **중앙 차분**을 씁니다.

$$f'(x) \approx \frac{f(x+h)-f(x-h)}{2h},\qquad h = 10^{-4}$$

**문제 6-1** 중앙 차분 수치 미분을 구현하세요.

> 🎬 **애니메이션** [D1-7 수치 미분 · 접선](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-7) · [D1-8 경사하강 등고선](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-8)  
> 📝 **워크시트** [6교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t6)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

#### 그림으로 보는 미분

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f6_secant.png" alt="할선이 접선으로 수렴" width="620"></p>

두 점을 잇는 직선(할선)의 기울기는 "평균 변화율"입니다. 두 점의 간격 h 를 줄이면 할선이 한 점에서 닿는 **접선**으로 다가가고, 그 기울기가 **미분값**입니다. 신경망에서는 "파라미터를 아주 조금 바꾸면 손실이 얼마나 바뀌나"를 이렇게 계산합니다.

In [ ]:
def numerical_diff(f, x, h=1e-4):
    return ____

In [ ]:
f = lambda x: x ** 2
assert np.isclose(numerical_diff(f, 3.0), 6.0)          # (x²)' = 2x
assert np.isclose(numerical_diff(np.sin, 0.0), 1.0)     # sin'(0) = cos 0 = 1
print("✅", numerical_diff(f, 3.0))

In [ ]:
#@title 🔒 정답 보기 6-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA2LTEg4pSA4pSACmRlZiBudW1lcmljYWxfZGlmZihmLCB4LCBoPTFlLTQpOgogICAgcmV0"
    "dXJuIChmKHggKyBoKSAtIGYoeCAtIGgpKSAvICgyICogaCk="
).decode("utf-8"))

**왜 h 를 1e-50 처럼 더 작게 하지 않을까?** — 부동소수점 반올림 오차 때문입니다.

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f6_h_error.png" alt="h 크기에 따른 수치 미분 오차" width="620"></p>

h 가 크면 곡선을 직선으로 근사한 **근사 오차**가, h 가 너무 작으면 float64 의 유효 자릿수(약 16자리) 때문에 **반올림 오차**가 커집니다. 중앙 차분은 근사 오차가 h² 에 비례해 훨씬 작으므로, h = 1e-4 에서 오차 1e-9 수준을 얻습니다.

In [ ]:
print(np.float32(1e-50))                 # 0.0 이 된다
for h in [1e-1, 1e-4, 1e-8, 1e-12, 1e-15]:
    print(f"h={h:.0e}  오차={abs(numerical_diff(f, 3.0, h) - 6.0):.2e}")

In [ ]:
# 접선 그리기: x=1 에서 f(x)=0.1x² + 0.5x 의 접선
f = lambda x: 0.1 * x ** 2 + 0.5 * x
x = np.linspace(-6, 6, 100)
x0 = 1.0
slope = numerical_diff(f, x0)
plt.plot(x, f(x), label="f(x)")
plt.plot(x, slope * (x - x0) + f(x0), "--", label=f"tangent, slope={slope:.2f}")
plt.scatter([x0], [f(x0)], c="red"); plt.legend(); plt.grid(alpha=.3); plt.show()

### 편미분과 기울기(gradient)

변수가 여러 개면 **하나씩만 움직이고 나머지는 고정**해 미분합니다(편미분). 편미분을 모은 벡터가 **기울기**입니다.

$$f(x_0, x_1) = x_0^2 + x_1^2 \quad\Rightarrow\quad \nabla f = \left(\frac{\partial f}{\partial x_0}, \frac{\partial f}{\partial x_1}\right) = (2x_0,\ 2x_1)$$

- 기울기는 **함수값이 가장 빨리 커지는 방향**을 가리킨다.
- 그러므로 $-\nabla f$ 방향으로 가면 함수값이 가장 빨리 작아진다 → **경사하강법**의 원리.

**문제 6-2** 임의 shape 의 배열 `x` 에 대해 기울기를 구하는 `numerical_gradient` 를 완성하세요. 이 함수는 **Day 3 역전파 검증(gradient check)에 그대로 다시 씁니다.**

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f6_gradient.png" alt="기울기 벡터와 최소점 방향" width="480"></p>

등고선에 **수직**으로 서 있는 화살표가 기울기입니다. 빨간 ∇f 는 산을 가장 가파르게 오르는 방향, 초록 −∇f 는 가장 가파르게 내려가는 방향입니다. 경사하강법은 매번 −∇f 방향으로 조금씩 이동해 골짜기 바닥(최소점)을 찾습니다.

#### numerical_gradient 의 동작 순서

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f6_numgrad.png" alt="numerical_gradient 5단계" width="760"></p>

In [ ]:
def numerical_gradient(f, x, h=1e-4):
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        orig = x[idx]
        x[idx] = orig + h;  fxh1 = f(x)      # f(x + h)
        x[idx] = ____;  fxh2 = f(x)      # f(x - h)
        grad[idx] = ____
        x[idx] = ____                        # 원래 값으로 되돌리기!
        it.iternext()
    return grad

In [ ]:
f2 = lambda x: np.sum(x ** 2)
x = np.array([3.0, 4.0])
g = numerical_gradient(f2, x)
assert np.allclose(g, [6.0, 8.0]) and np.allclose(x, [3.0, 4.0])
M = np.array([[1.0, 2.0], [3.0, 4.0]])
assert np.allclose(numerical_gradient(f2, M), 2 * M)     # 2차원 배열도 동작
print("✅", g)

In [ ]:
#@title 🔒 정답 보기 6-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA2LTIg4pSA4pSACmRlZiBudW1lcmljYWxfZ3JhZGllbnQoZiwgeCwgaD0xZS00KToKICAg"
    "IGdyYWQgPSBucC56ZXJvc19saWtlKHgpCiAgICBpdCA9IG5wLm5kaXRlcih4LCBmbGFncz1bIm11bHRpX2luZGV4"
    "Il0pCiAgICB3aGlsZSBub3QgaXQuZmluaXNoZWQ6CiAgICAgICAgaWR4ID0gaXQubXVsdGlfaW5kZXgKICAgICAg"
    "ICBvcmlnID0geFtpZHhdCiAgICAgICAgeFtpZHhdID0gb3JpZyArIGg7ICBmeGgxID0gZih4KSAgICAgICMgZih4"
    "ICsgaCkKICAgICAgICB4W2lkeF0gPSBvcmlnIC0gaDsgIGZ4aDIgPSBmKHgpICAgICAgIyBmKHggLSBoKQogICAg"
    "ICAgIGdyYWRbaWR4XSA9IChmeGgxIC0gZnhoMikgLyAoMiAqIGgpCiAgICAgICAgeFtpZHhdID0gb3JpZyAgICAg"
    "ICAgICAgICAgICAgICAgICAgICMg7JuQ656YIOqwkuycvOuhnCDrkJjrj4zrpqzquLAhCiAgICAgICAgaXQuaXRl"
    "cm5leHQoKQogICAgcmV0dXJuIGdyYWQKCvCfkqEg7ZW07ISkCnhbaWR4XSA9IG9yaWcg66GcIOuQmOuPjOumrOyn"
    "gCDslYrsnLzrqbQg64uk7J2MIOybkOyGjOulvCDrr7jrtoTtlaAg65WMIHgg6rCAIOydtOuvuCDrsJTrgIzslrQg"
    "7J6I7Ja0IOq4sOyauOq4sOqwgCDti4DrprDri6QuIO2ZleyduCDshYDsnbQgeCDqsIAg6re464yA66Gc7J247KeA"
    "64+EIOqygOyCrO2VmOuKlCDsnbTsnKDri6Qu"
).decode("utf-8"))

In [ ]:
# -기울기 벡터장: 화살표가 모두 최소점 (0, 0) 을 향한다
g0, g1 = np.meshgrid(np.linspace(-2, 2, 13), np.linspace(-2, 2, 13))
G = np.array([numerical_gradient(f2, np.array([a, b])) for a, b in zip(g0.ravel(), g1.ravel())])
plt.figure(figsize=(5, 5))
plt.contour(g0, g1, g0 ** 2 + g1 ** 2, levels=10, alpha=.4)
plt.quiver(g0.ravel(), g1.ravel(), -G[:, 0], -G[:, 1], angles="xy", color="tab:red")
plt.title("-gradient of x0² + x1²"); plt.axis("equal"); plt.show()

#### ⚠️ 자주 하는 실수

- **정수 배열**에 `numerical_gradient` 를 쓰면 `x[idx] = orig + h` 가 정수로 잘려 기울기가 0 이 됩니다 → `np.array([3.0, 4.0])` 처럼 실수로.
- `x[idx] = orig` 되돌리기를 빼먹으면 다음 원소의 기울기가 틀립니다.
- 기울기 방향으로 **더하면**(+) 손실이 커집니다. 경사하강은 **빼기**(−).

#### 📌 6교시 정리

- 미분 = h → 0 의 변화율, 컴퓨터에서는 **중앙 차분** `(f(x+h) − f(x−h)) / 2h`, h ≈ 1e-4.
- 기울기 = 편미분을 모은 벡터, 가장 빨리 커지는 방향 → −∇f 로 내려간다.
- `numerical_gradient` 는 Day 3 역전파 검증(gradient check)의 정답지.

## 7교시 — 확률 기초

신경망은 **난수로 가중치를 초기화**하고(Day 4), **확률로 답을 낸다**(Day 2 Softmax). 오늘은 필요한 만큼만 다룹니다.

| 개념 | 식 | NumPy |
|---|---|---|
| 평균 | $\mu = \frac{1}{N}\sum x_i$ | `x.mean()` |
| 분산 | $\sigma^2 = \frac{1}{N}\sum (x_i-\mu)^2$ | `x.var()` |
| 표준편차 | $\sigma$ | `x.std()` |
| 균등분포 $U(a,b)$ | 구간 안 모든 값이 같은 확률 | `rng.uniform(a, b, n)` |
| 정규분포 $N(\mu,\sigma^2)$ | 종 모양 | `rng.normal(mu, sigma, n)` |

> 🎬 **애니메이션** [D1-9 분포 · 표준화](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-9)  
> 📝 **워크시트** [7교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t7)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

#### 그림으로 보는 두 분포

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f7_dist.png" alt="균등분포와 정규분포 히스토그램" width="760"></p>

- **균등분포**: 구간 안 모든 값이 같은 확률 → 평평한 히스토그램. 가중치 초기화(Xavier uniform)에 씁니다.
- **정규분포**: 평균 근처 값이 가장 많고 멀어질수록 드물어지는 종 모양. ±1σ 안에 약 68%, ±2σ 안에 약 95% 가 들어갑니다. 측정 잡음, 가중치 초기화(He normal)에 씁니다.

In [ ]:
u = rng.uniform(-1, 1, 10_000)
n = rng.normal(0, 1, 10_000)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(u, bins=50); ax[0].set_title(f"Uniform(-1,1)  mean={u.mean():.3f}  std={u.std():.3f}")
ax[1].hist(n, bins=50); ax[1].set_title(f"Normal(0,1)  mean={n.mean():.3f}  std={n.std():.3f}")
plt.show()

**문제 7-1** 주사위를 10만 번 던져 각 눈이 나올 확률을 추정하세요. (`rng.integers(1, 7, n)` 은 1~6)

In [ ]:
dice = rng.integers(1, 7, 100_000)
prob = np.array([____ for k in range(1, 7)])

In [ ]:
assert prob.shape == (6,) and np.isclose(prob.sum(), 1) and np.allclose(prob, 1 / 6, atol=0.01)
print("✅", prob)

In [ ]:
#@title 🔒 정답 보기 7-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA3LTEg4pSA4pSACmRpY2UgPSBybmcuaW50ZWdlcnMoMSwgNywgMTAwXzAwMCkKcHJvYiA9"
    "IG5wLmFycmF5KFtucC5tZWFuKGRpY2UgPT0gaykgZm9yIGsgaW4gcmFuZ2UoMSwgNyldKQ=="
).decode("utf-8"))

**문제 7-2** **표준화** — 평균 0, 표준편차 1 로 맞추기. 입력 정규화와 Day 4 Batch Normalization 의 핵심 식입니다.

$$z = \frac{x - \mu}{\sigma}$$

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f7_standardize.png" alt="표준화 전후 히스토그램" width="760"></p>

표준화는 분포의 **모양은 그대로 두고** 위치(평균)와 폭(표준편차)만 맞춥니다. 키(cm)와 몸무게(kg)처럼 단위와 크기가 다른 입력을 같은 눈금으로 맞춰 주면 경사하강이 한쪽 방향으로만 크게 흔들리지 않아 학습이 안정됩니다 — 8교시에서 직접 보는 "방향마다 가파름이 다른" 문제와 같은 이야기입니다.

In [ ]:
def standardize(x):
    return ____

heights = rng.normal(170, 8, 1000)       # 키 데이터 (cm)
z = standardize(heights)

In [ ]:
assert np.isclose(z.mean(), 0, atol=1e-9) and np.isclose(z.std(), 1); print("✅", z[:5])

In [ ]:
#@title 🔒 정답 보기 7-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA3LTIg4pSA4pSACmRlZiBzdGFuZGFyZGl6ZSh4KToKICAgIHJldHVybiAoeCAtIHgubWVh"
    "bigpKSAvIHguc3RkKCkKCmhlaWdodHMgPSBybmcubm9ybWFsKDE3MCwgOCwgMTAwMCkgICAgICAgIyDtgqQg642w"
    "7J207YSwIChjbSkKeiA9IHN0YW5kYXJkaXplKGhlaWdodHMp"
).decode("utf-8"))

### log 와 exp — Day 2 손실 함수의 예고편

- `np.log` : 확률(0~1)을 손실로 바꿀 때 쓴다. **`log(0) = -inf`** 이므로 아주 작은 값 `eps` 를 더한다.
- `np.exp` : 점수를 확률로 바꿀 때 쓴다. **값이 크면 overflow** 가 난다.

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f7_exp_log.png" alt="exp 와 -log 그래프" width="760"></p>

In [ ]:
with np.errstate(divide="ignore", over="ignore"):
    print("log(0)        =", np.log(0.0))
    print("log(0 + 1e-7) =", np.log(0.0 + 1e-7))
    print("exp(1000)     =", np.exp(1000.0))       # inf → Day 2 에서 해결
p = np.linspace(0.01, 1, 100)
plt.plot(p, -np.log(p)); plt.xlabel("probability of correct class"); plt.ylabel("-log p")
plt.title("small probability → large loss"); plt.grid(alpha=.3); plt.show()

#### ⚠️ 자주 하는 실수

- `rng = np.random.default_rng(42)` 로 seed 를 고정하지 않으면 실행할 때마다 결과가 달라 비교가 어렵습니다.
- `x.std()` 는 모집단 표준편차(N 으로 나눔). 표본 표준편차(N−1)는 `x.std(ddof=1)`.
- `np.log(0)` → `-inf`, 이후 계산이 모두 `nan` 이 됩니다. 확률에는 작은 `eps` 를 더하세요.

#### 📌 7교시 정리

- 균등 · 정규분포 난수는 가중치 초기화의 재료, seed 로 재현성 확보.
- 표준화 `(x − μ) / σ` → 평균 0, 표준편차 1 (입력 정규화 · BatchNorm).
- `exp` 는 overflow, `log(0)` 은 −inf — Day 2 Softmax · Cross-Entropy 에서 해결.

## 8교시 — 미니 프로젝트: 경사하강법으로 최소점 찾기

목표 함수

$$f(x, y) = x^2 + 3y^2 \qquad (\text{최소점}: (0, 0),\ f = 0)$$

경사하강 갱신식 ($\eta$ = 학습률) — 6교시의 기울기를 이용해 $-\nabla f$ 방향으로 조금씩 내려갑니다.

$$\mathbf{p} \leftarrow \mathbf{p} - \eta\,\nabla f(\mathbf{p})$$

| 과제 | 내용 | 배점 |
|---|---|---|
| 1 | 목표 함수 · 해석적 기울기 구현 | 20 |
| 2 | 수치 기울기와 해석 기울기 비교 | 20 |
| 3 | `gradient_descent` 구현 | 30 |
| 4 | 학습률 4개로 경로 비교 그래프 | 20 |
| 5 | 결과 해석 (마크다운 답변) | 10 |

과제 끝의 **채점 셀**을 실행하면 자동 채점 점수가 나옵니다(과제 5 는 강사 채점). 제출은 채점 셀 결과가 보이는 상태로 이 노트북을 공유합니다.

> 🎬 **애니메이션** [D1-8 경사하강 등고선](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/animation.html#d1-8)  
> 📝 **워크시트** [8교시 개념 워크시트 (10문항)](https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/worksheet.html#t8)  
> 수업 전에 애니메이션으로 **예측**하고, 노트북 코드로 **확인**한 뒤, 워크시트로 **점검**합니다.

#### 미리 보는 경사하강 경로

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f8_gd_path.png" alt="경사하강 경로 (학습률 0.1)" width="620"></p>

출발점 p₀ = (−4, 2) 에서 기울기는 ∇f = (2·(−4), 6·2) = (−8, 12) 입니다. 학습률 0.1 을 곱해 빼면 p₁ = (−4, 2) − (−0.8, 1.2) = **(−3.2, 0.8)**. 이 계산을 반복하면 점들이 최소점 (0, 0) 으로 다가갑니다. y 방향(3y²)이 x 방향보다 가팔라서 y 가 먼저 빠르게 줄어드는 것도 보입니다. 과제에서 학습률을 바꿔 가며 이 경로가 어떻게 달라지는지 확인하세요.

### 과제 1 — 목표 함수와 해석적 기울기 (20점)

$\nabla f = (\partial f/\partial x,\ \partial f/\partial y) = (2x,\ 6y)$

In [ ]:
def f(p):
    x, y = p[0], p[1]
    return ____

def grad_f(p):
    x, y = p[0], p[1]
    return np.array([____, ____])

In [ ]:
#@title 🔒 정답 보기 8-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA4LTEg4pSA4pSACmRlZiBmKHApOgogICAgeCwgeSA9IHBbMF0sIHBbMV0KICAgIHJldHVy"
    "biB4ICoqIDIgKyAzICogeSAqKiAyCgpkZWYgZ3JhZF9mKHApOgogICAgeCwgeSA9IHBbMF0sIHBbMV0KICAgIHJl"
    "dHVybiBucC5hcnJheShbMiAqIHgsIDYgKiB5XSk="
).decode("utf-8"))

### 과제 2 — 수치 기울기와 비교 (20점)

6교시에서 만든 `numerical_gradient` 를 그대로 사용해 해석 기울기와의 최대 오차를 구하세요.

In [ ]:
p = np.array([1.5, -2.0])
max_err = ____
print("최대 오차:", max_err)

In [ ]:
#@title 🔒 정답 보기 8-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA4LTIg4pSA4pSACnAgPSBucC5hcnJheShbMS41LCAtMi4wXSkKbWF4X2VyciA9IG5wLm1h"
    "eChucC5hYnMobnVtZXJpY2FsX2dyYWRpZW50KGYsIHApIC0gZ3JhZF9mKHApKSkKcHJpbnQoIuy1nOuMgCDsmKTs"
    "sKg6IiwgbWF4X2Vycik="
).decode("utf-8"))

### 과제 3 — `gradient_descent` (30점)

- `init` 에서 출발해 `steps` 번 갱신한다.
- 반환값 `history` 는 출발점을 포함한 모든 위치, shape `(steps + 1, 2)`.
- 위치가 너무 커지면(노름 > 1e6) 발산으로 보고 멈춘다.

In [ ]:
def gradient_descent(grad, init, lr, steps):
    p = np.array(init, dtype=float)
    history = [p.copy()]
    for _ in range(steps):
        p = ____
        history.append(____)
        if np.linalg.norm(p) > 1e6:
            break
    return np.array(history)

In [ ]:
h = gradient_descent(grad_f, [-4.0, 2.0], lr=0.1, steps=100)
print("시작:", h[0], "→ 끝:", h[-1], " f =", f(h[-1]))

In [ ]:
#@title 🔒 정답 보기 8-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA4LTMg4pSA4pSACmRlZiBncmFkaWVudF9kZXNjZW50KGdyYWQsIGluaXQsIGxyLCBzdGVw"
    "cyk6CiAgICBwID0gbnAuYXJyYXkoaW5pdCwgZHR5cGU9ZmxvYXQpCiAgICBoaXN0b3J5ID0gW3AuY29weSgpXQog"
    "ICAgZm9yIF8gaW4gcmFuZ2Uoc3RlcHMpOgogICAgICAgIHAgPSBwIC0gbHIgKiBncmFkKHApCiAgICAgICAgaGlz"
    "dG9yeS5hcHBlbmQocC5jb3B5KCkpCiAgICAgICAgaWYgbnAubGluYWxnLm5vcm0ocCkgPiAxZTY6CiAgICAgICAg"
    "ICAgIGJyZWFrCiAgICByZXR1cm4gbnAuYXJyYXkoaGlzdG9yeSk="
).decode("utf-8"))

### 과제 4 — 학습률 비교 (20점)

학습률 `0.01, 0.1, 0.3, 0.35` 로 각각 30스텝 진행한 경로를 등고선 위에 그리세요.

In [ ]:
lrs = [0.01, 0.1, 0.3, 0.35]
paths = {lr: gradient_descent(grad_f, [-4.0, 2.0], lr=lr, steps=30) for lr in lrs}

gx, gy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for axi, lr in zip(ax, lrs):
    path = paths[lr]
    axi.contour(gx, gy, gx ** 2 + 3 * gy ** 2, levels=20, alpha=.4)
    axi.plot(____, "o-", ms=3, color="tab:red")
    axi.set_xlim(-5, 5); axi.set_ylim(-4, 4)
    axi.set_title(f"lr={lr}  final f={f(path[-1]):.2e}")
plt.show()

In [ ]:
#@title 🔒 정답 보기 8-4 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA4LTQg4pSA4pSACmxycyA9IFswLjAxLCAwLjEsIDAuMywgMC4zNV0KcGF0aHMgPSB7bHI6"
    "IGdyYWRpZW50X2Rlc2NlbnQoZ3JhZF9mLCBbLTQuMCwgMi4wXSwgbHI9bHIsIHN0ZXBzPTMwKSBmb3IgbHIgaW4g"
    "bHJzfQoKZ3gsIGd5ID0gbnAubWVzaGdyaWQobnAubGluc3BhY2UoLTUsIDUsIDIwMCksIG5wLmxpbnNwYWNlKC00"
    "LCA0LCAyMDApKQpmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKDEsIDQsIGZpZ3NpemU9KDE4LCA0KSkKZm9yIGF4aSwg"
    "bHIgaW4gemlwKGF4LCBscnMpOgogICAgcGF0aCA9IHBhdGhzW2xyXQogICAgYXhpLmNvbnRvdXIoZ3gsIGd5LCBn"
    "eCAqKiAyICsgMyAqIGd5ICoqIDIsIGxldmVscz0yMCwgYWxwaGE9LjQpCiAgICBheGkucGxvdChwYXRoWzosIDBd"
    "LCBwYXRoWzosIDFdLCAiby0iLCBtcz0zLCBjb2xvcj0idGFiOnJlZCIpCiAgICBheGkuc2V0X3hsaW0oLTUsIDUp"
    "OyBheGkuc2V0X3lsaW0oLTQsIDQpCiAgICBheGkuc2V0X3RpdGxlKGYibHI9e2xyfSAgZmluYWwgZj17ZihwYXRo"
    "Wy0xXSk6LjJlfSIpCnBsdC5zaG93KCk="
).decode("utf-8"))

### 과제 5 — 결과 해석 (10점)

아래 셀에 답을 쓰세요.

1. `lr=0.01` 은 왜 30스텝 안에 최소점에 닿지 못했나?
2. `lr=0.3` 에서 y 좌표가 지그재그로 움직이는 이유는? (힌트: y 갱신식 $y \leftarrow y - \eta\cdot 6y = (1-6\eta)\,y$)
3. `lr=0.35` 는 왜 발산하나? 발산하지 않는 학습률의 경계값은?

✍️ *여기에 답을 쓰세요.*

In [ ]:
#@title 🔒 해설 보기 8교시 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIO2VtOyEpCDilIDilIAKMS4g7ZWcIOuyiOyXkCDsm4Dsp4HsnbTripQg6rGw66asIM634oiHIGYg6rCA"
    "IOuEiOustCDsnpHslYQgMzDsiqTthZ3snLzroZzripQg67aA7KGx7ZWY64ukLiDtlZnsirXrpaDsnbQg7J6R7Jy8"
    "66m0IOyViOygleyggeydtOyngOunjCDripDrpqzri6QuCjIuIDEgLSA2w5cwLjMgPSAtMC44IOydtOudvCDrp6Qg"
    "7Iqk7YWdIHkg7J2YIOu2gO2YuOqwgCDrsJTrgJDri6Qo7KeE64+ZKS4g7YGs6riw64qUIDAuOOuwsOyUqSDspITs"
    "lrQg6rKw6rWtIOyImOugtO2VnOuLpC4KMy4gMSAtIDbDlzAuMzUgPSAtMS4xLCDtgazquLDqsIAg66ekIOyKpO2F"
    "nSAxLjHrsLDroZwg7Luk7KeE64ukLiDsiJjroLQg7KGw6rG0IHwxLTbOt3wgPCAxIOKHkiAwIDwgzrcgPCAxLzMg"
    "4omIIDAuMzMzLiB4IOuwqe2WpSDsobDqsbQozrc8MSnrs7Tri6QgeSDrsKntlqXsnbQg642UIOyXhOqyqe2VmOuv"
    "gOuhnCDqsIDsnqUg6rCA7YyM66W4IOuwqe2WpeydtCDtlZnsirXrpaAg7IOB7ZWc7J2EIOygle2VnOuLpCDigJQg"
    "RGF5IDQgTW9tZW50dW0gwrcgQWRhbSDsnZgg7Lac67Cc7KCQLg=="
).decode("utf-8"))

### 채점 셀

In [ ]:
score = 0
try:
    assert np.isclose(f(np.array([1.0, 2.0])), 13.0) and np.allclose(grad_f(np.array([1.0, 2.0])), [2.0, 12.0])
    score += 20; print("✅ 과제 1  +20")
except Exception as e: print("❌ 과제 1", e)
try:
    assert max_err < 1e-6
    score += 20; print("✅ 과제 2  +20")
except Exception as e: print("❌ 과제 2", e)
try:
    h = gradient_descent(grad_f, [-4.0, 2.0], lr=0.1, steps=100)
    assert h.shape == (101, 2) and np.allclose(h[0], [-4, 2]) and np.linalg.norm(h[-1]) < 1e-3
    assert np.linalg.norm(gradient_descent(grad_f, [-4.0, 2.0], lr=0.35, steps=200)[-1]) > 1e3
    score += 30; print("✅ 과제 3  +30")
except Exception as e: print("❌ 과제 3", e)
try:
    assert set(paths) == set(lrs) and all(len(paths[lr]) == 31 for lr in lrs)
    score += 20; print("✅ 과제 4  +20")
except Exception as e: print("❌ 과제 4", e)
print(f"\n자동 채점 {score} / 90  (과제 5 는 강사 채점 10점)")

### 🔎 더 알아보기 — 학습률과 수렴 조건

<p align="center"><img src="https://imguru-mooc.github.io/AI_CAMPUS/NN-day1/img/f8_lr.png" alt="학습률별 y 좌표 변화" width="640"></p>

과제 5 를 제출한 뒤에 보세요. y 좌표만 떼어 보면 매 스텝 `y ← (1 − 6η) y` 로 곱해집니다. 곱하는 수의 크기가 1 보다 작으면 줄어들고(수렴), 음수면 부호가 바뀌며 진동하고, 1 보다 크면 커집니다(발산). 실제 신경망은 방향이 수백만 개라 이렇게 손으로 풀 수 없으므로, Day 4 에서 **Momentum · Adam** 같은 옵티마이저로 방향마다 걸음 크기를 자동 조절합니다.

#### 📌 8교시 정리

- 경사하강: `p ← p − η ∇f(p)` 를 반복해 최소점을 찾는다.
- 학습률이 작으면 느리고, 적당하면 수렴, 크면 진동 · 발산.
- 가장 가파른 방향이 학습률의 상한을 정한다 → Day 4 옵티마이저.

## 오늘의 정리

### 셀프 체크

- [ ] `b = a` 와 `b = a.copy()` 의 차이를 설명할 수 있다
- [ ] `(3, 4)` 배열에 `sum(axis=0)`, `sum(axis=1, keepdims=True)` 를 하면 shape 가 어떻게 되는지 말할 수 있다
- [ ] 브로드캐스팅이 되는 shape 쌍과 안 되는 쌍을 구분할 수 있다
- [ ] `X @ W + b` 의 shape 를 추적할 수 있다
- [ ] 중앙 차분으로 미분하고, 기울기가 가리키는 방향을 설명할 수 있다
- [ ] 표준화 식을 쓰고 왜 쓰는지 말할 수 있다
- [ ] 학습률에 따라 수렴 · 진동 · 발산이 갈리는 이유를 $(1-6\eta)$ 로 설명할 수 있다

### 내일 (Day 2) 예고

퍼셉트론 → XOR → 활성화 함수 → 행렬 순전파 → Softmax · Cross-Entropy → 로지스틱 회귀. 오늘 만든 **브로드캐스팅 · `@` · `numerical_gradient` · `log`/`exp` · 경사하강**이 모두 다시 나옵니다.